# Playground Series — 表形式コンペ実践

毎月開催の表形式コンペ Playground Series を題材に、CV 構築から特徴量・アンサンブルまでを通します。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/playground-series`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Playground Series 形式（id, 数値特徴量, 二値 target）の合成データを準備し、
# "/kaggle/input/<competition>/..." のプレースホルダパスから透過的に読み込めるようにします。
import os
import numpy as np
import pandas as pd

def _ensure_playground_data():
    data_dir = os.path.abspath("./.kaggle_input/playground-series")
    os.makedirs(data_dir, exist_ok=True)
    orig_read_csv = pd.read_csv
    if not getattr(orig_read_csv, "_is_playground_wrapped", False):
        def _patched_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/<competition>/"):
                if not os.path.exists(filepath_or_buffer):
                    rel = filepath_or_buffer[len("/kaggle/input/<competition>/"):]
                    filepath_or_buffer = os.path.join(data_dir, rel)
            return orig_read_csv(filepath_or_buffer, *args, **kwargs)
        _patched_read_csv._is_playground_wrapped = True
        pd.read_csv = _patched_read_csv

    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")
    if os.path.exists(train_path) and os.path.exists(test_path):
        print(f"Using existing Playground Series dataset at: {data_dir}")
        return

    rng = np.random.default_rng(42)
    def _make_split(n, start_id, with_target):
        feats = {f"feat_{i}": rng.normal(0.0, 1.0, size=n).round(4) for i in range(8)}
        df = pd.DataFrame({"id": np.arange(start_id, start_id + n), **feats})
        if with_target:
            logit = (
                -0.8
                + 1.1 * df["feat_0"]
                - 0.8 * df["feat_1"]
                + 0.6 * df["feat_2"] * df["feat_3"]
                + 0.5 * (df["feat_4"] > 0).astype(float)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            df["target"] = (rng.random(n) < prob).astype(int)
        return df

    _make_split(1000, 0, True).to_csv(train_path, index=False)
    _make_split(400, 1000, False).to_csv(test_path, index=False)
    print(f"Playground Series sample dataset ready at: {data_dir}")

_ensure_playground_data()

## 概要

[Playground Series](https://www.kaggle.com/competitions?hostSegmentIdFilter=8) は、Kaggle が
毎月開催する**表形式の練習コンペ**です。実データに近い（合成された）データで、評価指標は回によって
AUC・RMSE などさまざま。入門（Titanic / House Prices）の次に、L4〜L6 の手法を統合的に試すのに最適です。

> **Note:** まず Overview と評価指標を確認します。指標に合わせて検証（交差検証）と後処理を設計するのが第一歩です。

## 1. EDA と検証設計

In [ ]:
import pandas as pd

train = pd.read_csv("/kaggle/input/<competition>/train.csv")
test = pd.read_csv("/kaggle/input/<competition>/test.csv")
train.shape, train["target"].value_counts(normalize=True)

目的変数の偏り・欠損・型を確認し、データ構造に合った分割（不均衡なら層化）で
**信頼できる 交差検証** を組みます。これが以降のすべての判断の土台です。

## 2. ベースライン（LightGBM + CV）

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = train.drop(columns=["id", "target"]); y = train["target"]
oof = np.zeros(len(X))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr, va in skf.split(X, y):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03)
    model.fit(X.iloc[tr], y.iloc[tr],
              eval_set=[(X.iloc[va], y.iloc[va])],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof[va] = model.predict_proba(X.iloc[va])[:, 1]
print("CV AUC:", roc_auc_score(y, oof))

勾配ブースティング（GBDT） と 早期終了 で素早く強いベースラインを作ります（L5）。

## 3. 改善

- 特徴量を一つずつ足して CV で確認（特徴量エンジニアリング、L4）
- 高カーディナリティは fold 内 ターゲットエンコーディング（リーク注意）
- ハイパラは Optuna で探索（L5）

## 4. アンサンブル

In [ ]:
# 同一の StratifiedKFold 分割を使って XGBoost と線形モデルの OOF 予測を作成します（fold 外への漏洩なし）
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

oof_lgbm = oof
oof_xgb = np.zeros(len(X))
oof_linear = np.zeros(len(X))
for tr, va in skf.split(X, y):
    m_xgb = XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, eval_metric="logloss", random_state=42
    )
    m_xgb.fit(X.iloc[tr], y.iloc[tr])
    oof_xgb[va] = m_xgb.predict_proba(X.iloc[va])[:, 1]

    m_lin = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m_lin.fit(X.iloc[tr], y.iloc[tr])
    oof_linear[va] = m_lin.predict_proba(X.iloc[va])[:, 1]

In [ ]:
# 系統の違うモデルの OOF を加重平均（重みは OOF で最適化）
blend = 0.5 * oof_lgbm + 0.3 * oof_xgb + 0.2 * oof_linear

In [ ]:
print("Blended CV AUC:", round(roc_auc_score(y, blend), 4))

LightGBM / XGBoost / 線形など多様なモデルを ブレンディング / スタッキング で統合します（L6）。
**重みは OOF で決め**、Public LB に合わせ込まない（シェイクアップ回避）。

## 関連トラック

- L4 特徴量エンジニアリング & 検証戦略
- L5 勾配ブースティング & ハイパラ最適化
- L6 アンサンブル & コンペワークフロー